# 01 Full-Data Baseline

Train a sequence classifier on the full AG News train split. This is the upper comparison point, not a distillation method.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from text_distillation.data import load_ag_news, make_tiny_subset
from text_distillation.data.datasets import get_label_names
from text_distillation.model import train_text_classifier
from text_distillation.saving import create_run_dir, save_experiment_config, save_metrics
from text_distillation.utils import get_git_commit_hash, set_seed

In [ ]:
EXPERIMENT_NAME = "full_data_agnews_bert_base"
DATASET_NAME = "ag_news"
MODEL_NAME = "bert-base-uncased"
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
MAX_LENGTH = 128
NUM_TRAIN_EPOCHS = 3.0
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
SEED = 42

# Keep None for the actual full-data baseline. Set small values for smoke checks.
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None

RUN_DIR = PROJECT_ROOT / "artifacts" / "runs" / EXPERIMENT_NAME
set_seed(SEED)

In [ ]:
dataset = load_ag_news()
train_dataset = dataset["train"]
test_dataset = dataset["test"]

if MAX_TRAIN_SAMPLES is not None:
    train_dataset = make_tiny_subset(train_dataset, total_size=MAX_TRAIN_SAMPLES, seed=SEED)
if MAX_EVAL_SAMPLES is not None:
    test_dataset = make_tiny_subset(test_dataset, total_size=MAX_EVAL_SAMPLES, seed=SEED)

label_names = get_label_names(dataset["train"], LABEL_COLUMN)
print(train_dataset)
print(test_dataset)
print(label_names)

In [ ]:
run_dir = create_run_dir(PROJECT_ROOT / "artifacts" / "runs", EXPERIMENT_NAME)
config = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset_name": DATASET_NAME,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "seed": SEED,
    "train_size": len(train_dataset),
    "eval_size": len(test_dataset),
    "git_commit": get_git_commit_hash(PROJECT_ROOT),
}
save_experiment_config(config, run_dir)

In [ ]:
model, metrics = train_text_classifier(
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    model_name=MODEL_NAME,
    output_dir=run_dir,
    text_column=TEXT_COLUMN,
    label_column=LABEL_COLUMN,
    num_labels=len(label_names),
    label_names=label_names,
    max_length=MAX_LENGTH,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    seed=SEED,
)
save_metrics(metrics, run_dir)
metrics